# Baseline Feature Importance → Train Models → Add EDA Features

Notebook này đi đúng thứ tự thí nghiệm:

1. Chạy baseline preprocessing từ dữ liệu clean chung.
2. Dùng Random Forest trên baseline để lấy đặc trưng quan trọng.
3. Train nhiều model với toàn bộ baseline features và top-K important features.
4. Tạo thêm đặc trưng từ EDA.
5. Train lại các model sau feature engineering.
6. So sánh tất cả kết quả trên cùng validation set và test model tốt nhất một lần cuối.

## 1. Import và cấu hình

In [1]:
from pathlib import Path
import sys

import joblib
import pandas as pd
from IPython.display import display
from sklearn.model_selection import train_test_split

CURRENT_DIR = Path.cwd().resolve()
if (CURRENT_DIR / "baseline_importance_then_eda_models.py").exists():
    NOTEBOOK_DIR = CURRENT_DIR
elif (CURRENT_DIR / "train" / "feature_engineering" / "baseline_importance_then_eda_models.py").exists():
    NOTEBOOK_DIR = CURRENT_DIR / "train" / "feature_engineering"
else:
    raise FileNotFoundError("Cannot find baseline_importance_then_eda_models.py")

sys.path.insert(0, str(NOTEBOOK_DIR))

import baseline_importance_then_eda_models as workflow

OUTPUT_DIR = workflow.OUTPUT_DIR
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Notebook dir:", NOTEBOOK_DIR)
print("Data path:", workflow.DATA_PATH)
print("Output dir:", OUTPUT_DIR)

Notebook dir: D:\HocTap\KT&XLTT\CUOIKI\train\feature_engineering
Data path: D:\HocTap\KT&XLTT\CUOIKI\data\diabetic_data_clean_common.csv
Output dir: D:\HocTap\KT&XLTT\CUOIKI\train\feature_engineering\baseline_importance_then_eda_outputs


## 2. Load dữ liệu và split

Split một lần duy nhất để mọi thí nghiệm dùng chung train/validation/test.

In [2]:
df = pd.read_csv(workflow.DATA_PATH)

y = df[workflow.TARGET_COLUMN]
X = df.drop(columns=[workflow.TARGET_COLUMN])

X_train_val, X_test, y_train_val, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=workflow.RANDOM_STATE,
    stratify=y,
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_val,
    y_train_val,
    test_size=0.20,
    random_state=workflow.RANDOM_STATE,
    stratify=y_train_val,
)

split_summary = pd.DataFrame({
    "split": ["train", "validation", "test"],
    "rows": [len(X_train), len(X_val), len(X_test)],
    "positive_rate": [y_train.mean(), y_val.mean(), y_test.mean()],
})
display(split_summary)

,split,rows,positive_rate
0,train,65128,0.460892
1,validation,16282,0.460877
2,test,20353,0.460915


## 3. Baseline preprocessing

Chưa thêm feature EDA ở bước này. Đây là dữ liệu nền để chọn đặc trưng quan trọng.

In [3]:
baseline_preprocessor, X_train_base, X_val_base, X_test_base, baseline_feature_names = workflow.prepare_baseline_data(
    X_train, X_val, X_test
)

pd.DataFrame({"feature_name": baseline_feature_names}).to_csv(
    OUTPUT_DIR / "baseline_encoded_feature_names.csv", index=False
)

print("Baseline train shape:", X_train_base.shape)
print("Baseline validation shape:", X_val_base.shape)
print("Baseline test shape:", X_test_base.shape)
display(pd.DataFrame({"feature_name": baseline_feature_names}).head(20))

Baseline train shape: (65128, 253)
Baseline validation shape: (16282, 253)
Baseline test shape: (20353, 253)


,feature_name
0,num__admission_type_id
1,num__discharge_disposition_id
2,num__admission_source_id
3,num__time_in_hospital
4,num__num_lab_procedures
5,num__num_procedures
6,num__num_medications
7,num__number_outpatient
8,num__number_emergency
9,num__number_inpatient


## 4. Chọn đặc trưng quan trọng bằng Random Forest baseline

Random Forest selector chỉ fit trên train baseline, sau đó lấy `feature_importances_`.

In [4]:
rf_selector, baseline_importance_df = workflow.fit_rf_feature_selector(
    X_train_base, y_train, baseline_feature_names
)

baseline_importance_df.to_csv(OUTPUT_DIR / "baseline_rf_feature_importance.csv", index=False)
joblib.dump(rf_selector, OUTPUT_DIR / "baseline_rf_feature_selector.joblib")

display(baseline_importance_df.head(30))

,rank,feature,importance
9,1,num__number_inpatient,0.083715
4,2,num__num_lab_procedures,0.068100
6,3,num__num_medications,0.063089
3,4,num__time_in_hospital,0.044702
1,5,num__discharge_disposition_id,0.043618
10,6,num__number_diagnoses,0.040592
5,7,num__num_procedures,0.030987
11,8,num__age_ordinal,0.027796
7,9,num__number_outpatient,0.023021
2,10,num__admission_source_id,0.022833


## 5. Train các model với baseline features

Gồm hai nhóm:

- `baseline_all_features`: dùng toàn bộ baseline features.
- `baseline_top_K_features`: dùng top-K feature quan trọng từ Random Forest baseline.

In [5]:
all_rows = []
all_trained_models = {}
test_matrices = {}

rows, models = workflow.train_and_evaluate_model_set(
    X_train_base,
    y_train,
    X_val_base,
    y_val,
    experiment="baseline",
    feature_set="all_features",
    output_dir=OUTPUT_DIR,
    tune_threshold=True,
)
all_rows.extend(rows)
for name, model in models.items():
    all_trained_models[("baseline", "all_features", name)] = model
test_matrices[("baseline", "all_features")] = X_test_base

for k in workflow.TOP_K_VALUES:
    top_idx = baseline_importance_df.head(k).index.to_numpy()
    top_feature_names = baseline_importance_df.head(k)["feature"].tolist()
    pd.DataFrame({"feature": top_feature_names}).to_csv(
        OUTPUT_DIR / f"baseline_top_{k}_features.csv", index=False
    )

    rows, models = workflow.train_and_evaluate_model_set(
        X_train_base[:, top_idx],
        y_train,
        X_val_base[:, top_idx],
        y_val,
        experiment="baseline_rf_selection",
        feature_set=f"top_{k}",
        output_dir=OUTPUT_DIR,
        tune_threshold=True,
    )
    all_rows.extend(rows)
    for name, model in models.items():
        all_trained_models[("baseline_rf_selection", f"top_{k}", name)] = model
    test_matrices[("baseline_rf_selection", f"top_{k}")] = X_test_base[:, top_idx]

baseline_results_df = pd.DataFrame(all_rows).sort_values(
    ["f1_score", "roc_auc", "accuracy"], ascending=False
)
display(baseline_results_df)

c:\Users\ADMIN\anaconda3\envs\A3Net\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
c:\Users\ADMIN\anaconda3\envs\A3Net\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
c:\Users\ADMIN\anaconda3\envs\A3Net\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
c:\Users\ADMIN\anaconda3\envs\A3Net\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning

,experiment,feature_set,model,threshold_strategy,threshold,accuracy,precision,recall,f1_score,roc_auc,train_time_sec
27,baseline_rf_selection,top_150,Random Forest,tuned_for_validation_f1,0.370974,0.584203,0.529085,0.889659,0.663552,0.701151,4.932706
21,baseline_rf_selection,top_100,Random Forest,tuned_for_validation_f1,0.366753,0.583221,0.528451,0.888593,0.662757,0.698349,9.401758
15,baseline_rf_selection,top_50,Random Forest,tuned_for_validation_f1,0.378895,0.595934,0.538648,0.859009,0.662113,0.694018,3.178268
3,baseline,all_features,Random Forest,tuned_for_validation_f1,0.373077,0.574745,0.522404,0.901119,0.661385,0.699693,6.209604
9,baseline_rf_selection,top_30,Random Forest,tuned_for_validation_f1,0.337262,0.566270,0.516654,0.913646,0.660056,0.690971,3.050902
1,baseline,all_features,Logistic Regression,tuned_for_validation_f1,0.376924,0.552696,0.508431,0.888060,0.646645,0.671219,111.899899
4,baseline,all_features,SVM,tuned_for_validation_f1,-0.257999,0.541764,0.501581,0.908715,0.646381,0.669809,7.348315
28,baseline_rf_selection,top_150,SVM,tuned_for_validation_f1,-0.254719,0.541764,0.501581,0.908715,0.646381,0.669635,3.765099
22,baseline_rf_selection,top_100,SVM,tuned_for_validation_f1,-0.230168,0.552328,0.508202,0.887660,0.646354,0.668721,2.671261
25,baseline_rf_selection,top_150,Logistic Regression,tuned_for_validation_f1,0.375900,0.550424,0.506976,0.890991,0.646240,0.670644,85.699103


## 6. Tạo thêm đặc trưng từ EDA

Sau khi đã có baseline feature importance và baseline model results, bước này mới thêm đặc trưng EDA.

In [6]:
fe_params, eda_preprocessor, X_train_eda, X_val_eda, X_test_eda, eda_feature_names = workflow.prepare_eda_data(
    X_train, X_val, X_test
)

pd.DataFrame([fe_params]).to_csv(OUTPUT_DIR / "eda_feature_engineering_params.csv", index=False)
pd.DataFrame({"feature_name": eda_feature_names}).to_csv(OUTPUT_DIR / "eda_encoded_feature_names.csv", index=False)
joblib.dump(eda_preprocessor, OUTPUT_DIR / "eda_preprocessor.joblib")

new_features = [
    "total_prior_visits",
    "has_prior_inpatient",
    "meds_per_day",
    "labs_per_day",
    "is_long_stay",
    "is_senior",
    "a1c_abnormal",
    "insulin_changed",
    "num_diabetes_drugs_used",
    "num_unique_diag_groups",
    "prior_inpatient_long_stay",
    "insulin_change_a1c_abnormal",
    "high_medication_load",
    "high_lab_load",
]
pd.DataFrame({"new_feature": new_features}).to_csv(OUTPUT_DIR / "eda_new_features.csv", index=False)

print("EDA train shape:", X_train_eda.shape)
display(pd.DataFrame([fe_params]))
display(pd.DataFrame({"new_feature": new_features}))

EDA train shape: (65128, 267)


,long_stay_q3,high_med_q3,high_lab_q3
0,6.0,20.0,57.0


,new_feature
0,total_prior_visits
1,has_prior_inpatient
2,meds_per_day
3,labs_per_day
4,is_long_stay
5,is_senior
6,a1c_abnormal
7,insulin_changed
8,num_diabetes_drugs_used
9,num_unique_diag_groups


## 7. Train lại các model sau khi thêm EDA features

In [7]:
rows, models = workflow.train_and_evaluate_model_set(
    X_train_eda,
    y_train,
    X_val_eda,
    y_val,
    experiment="eda_features",
    feature_set="all_features",
    output_dir=OUTPUT_DIR,
    tune_threshold=True,
)
all_rows.extend(rows)
for name, model in models.items():
    all_trained_models[("eda_features", "all_features", name)] = model
test_matrices[("eda_features", "all_features")] = X_test_eda

final_comparison_df = pd.DataFrame(all_rows).sort_values(
    ["f1_score", "roc_auc", "accuracy"], ascending=False
)
final_comparison_df.insert(0, "rank", range(1, len(final_comparison_df) + 1))
final_comparison_df.to_csv(OUTPUT_DIR / "baseline_importance_then_eda_validation_comparison.csv", index=False)

display(final_comparison_df)

c:\Users\ADMIN\anaconda3\envs\A3Net\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
c:\Users\ADMIN\anaconda3\envs\A3Net\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


,rank,experiment,feature_set,model,threshold_strategy,threshold,accuracy,precision,recall,f1_score,roc_auc,train_time_sec
27,1,baseline_rf_selection,top_150,Random Forest,tuned_for_validation_f1,0.370974,0.584203,0.529085,0.889659,0.663552,0.701151,4.932706
21,2,baseline_rf_selection,top_100,Random Forest,tuned_for_validation_f1,0.366753,0.583221,0.528451,0.888593,0.662757,0.698349,9.401758
15,3,baseline_rf_selection,top_50,Random Forest,tuned_for_validation_f1,0.378895,0.595934,0.538648,0.859009,0.662113,0.694018,3.178268
3,4,baseline,all_features,Random Forest,tuned_for_validation_f1,0.373077,0.574745,0.522404,0.901119,0.661385,0.699693,6.209604
9,5,baseline_rf_selection,top_30,Random Forest,tuned_for_validation_f1,0.337262,0.566270,0.516654,0.913646,0.660056,0.690971,3.050902
33,6,eda_features,all_features,Random Forest,tuned_for_validation_f1,0.370428,0.580948,0.527160,0.880730,0.659548,0.696378,5.024562
34,7,eda_features,all_features,SVM,tuned_for_validation_f1,-0.195617,0.576833,0.525481,0.843683,0.647606,0.671280,12.286910
31,8,eda_features,all_features,Logistic Regression,tuned_for_validation_f1,0.404574,0.583159,0.530581,0.828891,0.647007,0.672178,112.125421
1,9,baseline,all_features,Logistic Regression,tuned_for_validation_f1,0.376924,0.552696,0.508431,0.888060,0.646645,0.671219,111.899899
4,10,baseline,all_features,SVM,tuned_for_validation_f1,-0.257999,0.541764,0.501581,0.908715,0.646381,0.669809,7.348315


## 8. Lấy feature importance sau EDA

Phần này dùng Random Forest đã train trên EDA all features để xem các feature mới có lọt top hay không.

In [8]:
eda_rf_model = all_trained_models[("eda_features", "all_features", "Random Forest")]
eda_importance_df = pd.DataFrame({
    "feature": eda_feature_names,
    "importance": eda_rf_model.feature_importances_,
}).sort_values("importance", ascending=False)
eda_importance_df.insert(0, "rank", range(1, len(eda_importance_df) + 1))
eda_importance_df.to_csv(OUTPUT_DIR / "eda_rf_feature_importance.csv", index=False)

display(eda_importance_df.head(30))

,rank,feature,importance
15,1,num__labs_per_day,0.056985
4,2,num__num_lab_procedures,0.054322
14,3,num__meds_per_day,0.053798
6,4,num__num_medications,0.048935
12,5,num__total_prior_visits,0.047925
9,6,num__number_inpatient,0.038500
1,7,num__discharge_disposition_id,0.035096
10,8,num__number_diagnoses,0.031919
3,9,num__time_in_hospital,0.030684
13,10,num__has_prior_inpatient,0.027835


## 9. Test set cho model tốt nhất

Chọn model có validation F1-score cao nhất, sau đó đánh giá test set một lần.

In [9]:
best_row = final_comparison_df.iloc[0].to_dict()
best_key = (best_row["experiment"], best_row["feature_set"], best_row["model"])
best_model = all_trained_models[best_key]
best_test_matrix = test_matrices[(best_row["experiment"], best_row["feature_set"])]

test_result = workflow.evaluate_best_on_test(best_row, best_model, best_test_matrix, y_test, OUTPUT_DIR)
joblib.dump(best_model, OUTPUT_DIR / "best_validation_model.joblib")

display(pd.DataFrame([test_result]))

,selected_by,experiment,feature_set,model,threshold_strategy,threshold,accuracy,precision,recall,f1_score,roc_auc
0,best_validation_f1,baseline_rf_selection,top_150,Random Forest,tuned_for_validation_f1,0.370974,0.583206,0.528468,0.888498,0.662744,0.70239


## 10. File output chính

- `baseline_rf_feature_importance.csv`: đặc trưng quan trọng từ baseline Random Forest.
- `baseline_top_30_features.csv`, `baseline_top_50_features.csv`, `baseline_top_100_features.csv`, `baseline_top_150_features.csv`: top-K đặc trưng baseline.
- `baseline_importance_then_eda_validation_comparison.csv`: bảng so sánh chính.
- `eda_new_features.csv`: danh sách đặc trưng tạo thêm từ EDA.
- `eda_rf_feature_importance.csv`: đặc trưng quan trọng sau khi thêm EDA features.
- `final_test_metrics.csv`: kết quả test set của model tốt nhất theo validation.